In [0]:
# Configure Azure Data Lake Storage Gen2 access key for the storage account
# This allows Spark to read/write data from the bronze, silver, and gold containers
spark.conf.set(
    "fs.azure.account.key.<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net",
    "<YOUR_AZURE_STORAGE_ACCESS_KEY>"
)

In [0]:
# List all files in the bronze/banking directory
files = dbutils.fs.ls("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>/")

# Filter to only CSV files
csv_files = [f.path for f in files if f.path.endswith(('.csv', '.CSV'))]
files_list = []

# Dynamically create DataFrames for each CSV file
for file_path in csv_files:
    # Extract the file name without extension for the variable name
    file_name = file_path.split('/')[-1]
    if file_name.endswith(('.csv', '.CSV')):
        file_name = file_name[:-4]
    files_list.append(file_name)
    
    # Replace invalid characters for variable names with underscores
    # Ensures variable names are valid Python identifiers
    var_name = ''.join([c if c.isalnum() else '_' for c in file_name])
    if var_name and var_name[0].isdigit():
        var_name = '_' + var_name
    
    # Read the CSV file into a DataFrame with header and schema inference
    df = (spark.read
          .format("csv")
          .option("header", True)
          .option("inferSchema", True)
          .load(file_path))
    
    # Assign the DataFrame to a variable in the global namespace
    # This creates variables like amex_history, chase_history, etc.
    globals()[var_name] = df

In [0]:
from functools import reduce

# Identify files that belong to Capital One (cone*) and Apple (apple*) accounts as they have transactions seperated in different statements
cone = []
apple_hist = []
for f in files_list:
    if f.startswith('cone'):
        cone.append(f)
    if f.startswith('apple'):
        apple_hist.append(f)

# Combine multiple Capital One CSV files into a single DataFrame
# Uses reduce to union all cone* DataFrames together
capital_history = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), [globals()[name] for name in cone if hasattr(globals()[name], 'unionByName')])

# Combine multiple Apple CSV files into a single DataFrame
apple_history = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), [globals()[name] for name in apple_hist if hasattr(globals()[name], 'unionByName')])

In [0]:
# Create a list of all account DataFrames, excluding individual cone* and apple* files
# since they've been combined into capital_history and apple_history
filtered_statements = [item for item in files_list if not item.startswith(('cone', 'apple'))]

# Add the combined DataFrames to the list
filtered_statements.append('capital_history')
filtered_statements.append('apple_history')

In [0]:
# Display the final list of account DataFrames to be processed
filtered_statements

In [0]:
# Remove redundant or unnecessary columns from each account's DataFrame
# This standardizes the schema and removes duplicate date/type columns
bank_history = bank_history.drop('Check#', 'Transaction Type')
chase_history = chase_history.drop('Post Date', 'Type', 'Memo')
discover_history = discover_history.drop('Trans. Date')
apple_history = apple_history.drop('Transaction Date', 'Merchant', 'Type', 'Purchased By')
capital_history = capital_history.drop('Posted Date', 'Card No.')

In [0]:
from pyspark.sql.functions import coalesce, col, when

# Capital One has separate credit and debit columns
# Combine them into a single 'amount' column:
# - Credits are positive values
# - Debits are negative values (hence the -col("debit"))
capital_history = capital_history.withColumn(
    "amount",
    coalesce(col("credit"), -col("debit"))
).drop("credit", "debit")

In [0]:
from pyspark.sql.functions import col, when

# Standardize amount columns across credit card accounts
# Credit card statements show charges as positive, but we want debits as negative
# Flip the sign by multiplying by -1 to match banking convention

# Amex: rename "Amount" to "amount" and flip sign
amex_history = amex_history.withColumnRenamed("Amount", "amount")
amex_history = amex_history.withColumn(
    "amount",
    -col("amount").cast("double")
)

# Discover: rename "Amount" to "amount" and flip sign
discover_history = discover_history.withColumnRenamed("Amount", "amount")
discover_history = discover_history.withColumn(
    "amount",
    -col("amount").cast("double")
)

# Apple: rename "Amount (USD)" to "amount" and flip sign
apple_history = apple_history.withColumnRenamed("Amount (USD)", "amount")
apple_history = apple_history.withColumn(
    "amount",
    -col("amount").cast("double")
)

In [0]:
from pyspark.sql.functions import coalesce, col, when, regexp_replace

# Bank history has separate Debits(-) and Credits(+) columns
# Combine them into a single 'amount' column using coalesce
bank_history = bank_history.withColumn(
    "amount",
    coalesce(col("Debits(-)"), col("Credits(+)"))
).drop("Debits(-)", "Credits(+)")

# Remove the dollar sign ($) prefix from amount values
bank_history = bank_history.withColumn(
    "amount",
    regexp_replace(col("amount"), r"^\$", "")
)

# Define a function to add a 'type' column (debit or credit)
# based on whether the amount is negative or positive
def add_type_col(df):
    """Add a 'type' column to classify transactions as debit or credit based on amount sign."""
    return df.withColumn(
        "type",
        when(col("amount").cast("string").startswith("-"), "debit").otherwise("credit")
    )

# Apply the type classification to all account DataFrames
amex_history_final = add_type_col(amex_history)
chase_history_final = add_type_col(chase_history)
discover_history_final = add_type_col(discover_history)
capital_history_final = add_type_col(capital_history)
apple_history_final = add_type_col(apple_history)
bank_history_final = add_type_col(bank_history)

In [0]:
from pyspark.sql.functions import when, col

# Fill null category values with "Payments and Credits" for Amex and Chase
# These accounts sometimes have missing categories for payment transactions
amex_history_final = amex_history_final.withColumn(
    "category",
    when(col("category").isNull(), "Payments and Credits").otherwise(col("category"))
)

chase_history_final = chase_history_final.withColumn(
    "category",
    when(col("category").isNull(), "Payments and Credits").otherwise(col("category"))
)

In [0]:
# Convert all bank_history column names to lowercase for consistency
# This ensures uniform column naming across all DataFrames
bank_history_final = bank_history_final.toDF(*[c.lower() for c in bank_history_final.columns])

# Remove the dollar sign ($) prefix from balance values
bank_history_final = bank_history_final.withColumn(
    "balance",
    regexp_replace(col("balance"), r"^\$", ""))

In [0]:
# Display all cleaned and standardized DataFrames for verification
# Each DataFrame now has consistent schema: date, description, amount, category, type
display(amex_history_final)
display(chase_history_final)
display(discover_history_final)
display(capital_history_final)
display(apple_history_final)
display(bank_history_final)

In [0]:
import sys

# Get all variables in the current global scope that end with 'final' and are DataFrames
# This dynamically identifies all the cleaned account DataFrames
final_dfs = [name for name, val in globals().items() if name.endswith('final') and 'DataFrame' in str(type(val))]

# Display the list of final DataFrames
final_dfs

# Saving Cleaned tables to Silver container

In [0]:
# Write all cleaned DataFrames to the silver container in Parquet format
# Silver layer contains cleaned and standardized data ready for further processing
# Using overwrite mode to replace any existing data
amex_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
chase_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
discover_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
capital_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
apple_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
bank_history_final.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")

In [0]:
# Read the cleaned DataFrames from the silver container
# This allows us to restart processing from the silver layer without re-running the cleaning steps
amex_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
chase_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
discover_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
capital_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
apple_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
bank_history_final = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")

In [0]:
from pyspark.sql.functions import lit, col

# Standardize column names across all credit card accounts and add an 'account' identifier
# This prepares each DataFrame for union by ensuring consistent schema

# Amex: standardize columns and add account identifier
amex_df = amex_history_final.select(
    col("Date").alias("date"),
    col("Description").alias("description"),
    col("amount"),
    col("category"),
    col("type")
).withColumn("account", lit("amex"))

# Chase: standardize columns and add account identifier
chase_df = chase_history_final.select(
    col("Transaction Date").alias("date"),
    col("Description").alias("description"),
    col("Amount").alias("amount"),
    col("category"),
    col("type")
).withColumn("account", lit("chase"))

# Discover: standardize columns and add account identifier
discover_df = discover_history_final.select(
    col("Post Date").alias("date"),
    col("Description").alias("description"),
    col("amount"),
    col("Category").alias("category"),
    col("type")
).withColumn("account", lit("discover"))

# Capital One: standardize columns and add account identifier
capital_df = capital_history_final.select(
    col("Transaction Date").alias("date"),
    col("Description").alias("description"),
    col("amount"),
    col("Category").alias("category"),
    col("type")
).withColumn("account", lit("capital"))

# Apple: standardize columns and add account identifier
apple_df = apple_history_final.select(
    col("Clearing Date").alias("date"),
    col("Description").alias("description"),
    col("amount"),
    col("Category").alias("category"),
    col("type")
).withColumn("account", lit("apple"))

# Combine all credit card accounts into a single DataFrame
# allowMissingColumns=True handles any schema differences gracefully
combined_nonbank_df = amex_df.unionByName(chase_df, allowMissingColumns=True) \
    .unionByName(discover_df, allowMissingColumns=True) \
    .unionByName(capital_df, allowMissingColumns=True) \
    .unionByName(apple_df, allowMissingColumns=True)

# Convert all column names to lowercase for consistency
combined_nonbank_df = combined_nonbank_df.toDF(*[c.lower() for c in combined_nonbank_df.columns])

display(combined_nonbank_df)

In [0]:
# Display the bank account DataFrame for verification before combining with credit cards
display(bank_history_final)

In [0]:
from pyspark.sql.functions import lit

# Add a 'balance' column to the combined credit card DataFrame
# Set to None since credit card transactions don't track running balance
# This ensures schema compatibility with bank_history which has a balance column
combined_nonbank_df = combined_nonbank_df.withColumn("balance", lit(None))

In [0]:
# Display the combined credit card DataFrame with the newly added balance column
display(combined_nonbank_df)

In [0]:
from pyspark.sql.functions import when, col, lower

# Categorize bank transactions based on description patterns
# Uses pattern matching on lowercased descriptions to assign categories
bank_history_category = bank_history_final.withColumn(
    "category",
    # Zelle and online transfers
    when(lower(col("description")).like("%zelle%") | lower(col("description")).like("%phone/online%"), "Zelle Transfers")
    # Credit card payments
    .when(lower(col("description")).like("%discover%") | lower(col("description")).like("%chase credit%") | lower(col("description")).like("%apple%")  | lower(col("description")).like("%amex%")  | lower(col("description")).like("%capital%") , "Credit Card Payments")
    # Auto loan payments
    .when(lower(col("description")).like("%nissan%"), "Auto Payments")
    # Housing and utility payments
    .when(lower(col("description")).like("%rent%") | lower(col("description")).like("%twin%") | lower(col("description")).like("%resident%") | lower(col("description")).like("%propert%") | lower(col("description")).like("%billpay%") | lower(col("description")).like("%og&e%") | lower(col("description")).like("%city%") | lower(col("description")).like("%att%"), "Housing and Utilities")
    # Tax payments
    .when(lower(col("description")).like("%tax%"), "Tax")
    # Investment and stock transactions
    .when(lower(col("description")).like("%fid%") | lower(col("description")).like("%robin%"), "Investment & Stocks")
    # Education expenses
    .when(lower(col("description")).like("%bursar%"), "Education")
    # General purchases and refunds
    .when(lower(col("description")).like("%merchant%") | lower(col("description")).like("%purchase%") | lower(col("description")).like("%paypal%"), "Purchases and Refunds")
    # Income deposits
    .when(lower(col("description")).like("%deposit%") | lower(col("description")).like("%check%") | lower(col("description")).like("%payroll%"), "Income")
    # Everything else
    .otherwise("Others")
)

display(bank_history_category)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Category consolidation mapping based on the 63 distinct categories from credit card data
# Maps granular categories to broader, standardized categories for analysis
CATEGORY_MAP = {
    # Transportation (13 categories consolidated)
    "Transportation-Fuel": "Transportation",
    "Gas": "Transportation",
    "Gasoline": "Transportation",
    "Gas/Automotive": "Transportation",
    "Automotive": "Transportation",
    "Other Travel": "Transportation",
    "Airfare": "Transportation",
    "Transportation": "Transportation",
    
    # Food & Dining (11 categories consolidated)
    "Restaurant-Bar & Café": "Food & Dining",
    "Restaurant-Restaurant": "Food & Dining",
    "Food & Drink": "Food & Dining",
    "Groceries": "Food & Dining",
    "Merchandise & Supplies-Groceries": "Food & Dining",
    "Restaurants": "Food & Dining",
    "Supermarkets": "Food & Dining",
    "Dining": "Food & Dining",
    "Grocery": "Food & Dining",
    "Alcohol": "Food & Dining",
    
    # Purchases and Refunds (15 categories consolidated - Shopping + Professional Services)
    "Merchandise & Supplies-General Retail": "Purchases and Refunds",
    "Merchandise & Supplies-Computer Supplies": "Purchases and Refunds",
    "Merchandise & Supplies-Mail Order": "Purchases and Refunds",
    "Merchandise & Supplies-Electronics Stores": "Purchases and Refunds",
    "Merchandise & Supplies-Wholesale Stores": "Purchases and Refunds",
    "Merchandise & Supplies-Internet Purchase": "Purchases and Refunds",
    "Shopping": "Purchases and Refunds",
    "Merchandise": "Purchases and Refunds",
    "Warehouse Clubs": "Purchases and Refunds",
    "Business Services-Other Services": "Purchases and Refunds",
    "Business Services-Office Supplies": "Purchases and Refunds",
    "Business Services-Internet Services": "Purchases and Refunds",
    "Business Services-Professional Services": "Purchases and Refunds",
    "Professional Services": "Purchases and Refunds",
    "Services": "Purchases and Refunds",
    "Government Services": "Purchases and Refunds",
    "Other Services": "Purchases and Refunds",
    "Communications-Other Telecom": "Purchases and Refunds",
    
    # Housing & Utilities (4 categories consolidated)
    "Home": "Housing & Utilities",
    "Bills & Utilities": "Housing & Utilities",
    "Phone/Cable": "Housing & Utilities",
    "Internet": "Housing & Utilities",
    
    # Health & Insurance (4 categories consolidated)
    "Health & Wellness": "Health & Insurance",
    "Medical Services": "Health & Insurance",
    "Business Services-Health Care Services": "Health & Insurance",
    "Health Care": "Health & Insurance",
    "Insurance": "Health & Insurance",
    
    # Entertainment & Travel (3 categories consolidated)
    "Entertainment": "Entertainment & Travel",
    "Travel": "Entertainment & Travel",
    "Travel/ Entertainment": "Entertainment & Travel",
    
    # Credit Due Payments (9 categories consolidated - Fees & Payments)
    "Fees & Adjustments-Fees & Adjustments": "Credit Due Payments",
    "Fees & Adjustments": "Credit Due Payments",
    "Payments and Credits": "Credit Due Payments",
    "Awards and Rebate Credits": "Credit Due Payments",
    "Fee/Interest Charge": "Credit Due Payments",
    "Payment/Credit": "Credit Due Payments",
    "Installment": "Credit Due Payments",
    "Payment": "Credit Due Payments",
    "Credit": "Credit Due Payments",
    "Debit": "Credit Due Payments",
    
    # Others (4 categories consolidated)
    "Other-Miscellaneous": "Others",
    "Personal": "Others",
    "Gifts & Donations": "Others",
    "Other": "Others",
    
    # Education (keep separate)
    "Education": "Education",
}

# Create UDF to map categories using the consolidation mapping
def map_category(cat):
    """Map granular categories to consolidated categories. Returns original if not in map."""
    if cat is None:
        return None
    return CATEGORY_MAP.get(cat, cat)  # Return original if not in map

map_category_udf = udf(map_category, StringType())

# Apply consolidation to combined_nonbank_df
combined_nonbank_df_consolidated = combined_nonbank_df.withColumn(
    "category", 
    map_category_udf(col("category"))
)

display(combined_nonbank_df_consolidated)

In [0]:
from pyspark.sql.functions import lit

# Add account identifier to bank history
bank_history_category_with_account = bank_history_category.withColumn("account", lit("bank"))

# Combine bank and credit card data into final static dataset
# This creates a unified view of all financial transactions across all accounts
final_static_data = combined_nonbank_df_consolidated.unionByName(bank_history_category_with_account, allowMissingColumns=True)

display(final_static_data)

#Saving final Gold Data Set

In [0]:
# Write the final consolidated dataset to the gold container
# Gold layer contains business-ready, fully processed data for analytics and reporting
final_static_data.write.mode("overwrite").parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")

In [0]:
# Read the final consolidated dataset from the gold container
# This allows analysis and reporting without re-running the entire pipeline
final_static_data = spark.read.parquet("abfss://<YOUR_AZURE_STORAGE_ACCOUNT>.dfs.core.windows.net/<YOUR_CONTAINER_NAME>")
display(final_static_data)